# Weather analysis: Ho Chi Minh City vs Tokyo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cassandrautils import getWeatherDF

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 30)

## 1. Load and prepare the data

* OpenWeatherMap returns temperatures in **Kelvin** by default, so we convert them to °C.
* `forecastdate` is the producer's report time. The producer container runs on UTC, so we also compute each
  city's **local time** (HCMC = UTC+7, Tokyo = UTC+9), which the daily-cycle analysis needs.

In [ ]:
weather = getWeatherDF()
assert weather is not None and not weather.empty, "weatherreport is empty - is the weather producer + sink running?"

weather["forecastdate"] = pd.to_datetime(weather["forecastdate"])
weather = weather.sort_values("forecastdate").reset_index(drop=True)

for col in ["temp", "feels_like", "temp_min", "temp_max"]:
    if weather[col].mean() > 200:          # still in Kelvin
        weather[col] = weather[col] - 273.15

UTC_OFFSET = {"Ho Chi Minh City": 7, "Tokyo": 9}
weather["local_time"] = weather["forecastdate"] + pd.to_timedelta(weather["location"].map(UTC_OFFSET).fillna(0), unit="h")
weather["local_hour"] = weather["local_time"].dt.hour
weather["feels_gap"] = weather["feels_like"] - weather["temp"]

CITY_COLORS = {"Ho Chi Minh City": "#d1495b", "Tokyo": "#00798c"}
weather.head()

In [ ]:
summary = weather.groupby("location").agg(
    records=("temp", "size"),
    first_report=("forecastdate", "min"),
    last_report=("forecastdate", "max"),
    avg_temp_c=("temp", "mean"),
    min_temp_c=("temp", "min"),
    max_temp_c=("temp", "max"),
    avg_humidity=("humidity", "mean"),
    avg_wind_ms=("wind", "mean"),
)
num_cols = summary.select_dtypes("number").columns
summary[num_cols] = summary[num_cols].round(2)
summary

## 2. Temperature over time

How different are the two climates, and how much does each city change during the period we collected?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for city, g in weather.groupby("location"):
    ax.plot(g["forecastdate"], g["temp"], label=city, color=CITY_COLORS.get(city), linewidth=1.8)
ax.set_title("Temperature over time")
ax.set_xlabel("Report time (UTC)")
ax.set_ylabel("Temperature (°C)")
ax.legend()
plt.show()

means = weather.groupby("location")["temp"].mean()
if len(means) == 2:
    print("Average temperature difference: {:.1f} °C ({} warmer)".format(abs(means.diff().iloc[-1]), means.idxmax()))

## 3. "Feels like" vs actual temperature

The *feels-like* temperature accounts for humidity and wind. A **positive gap** means the air feels hotter than
the thermometer shows, which is typical of humid tropical cities.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=weather, x="location", y="feels_gap", hue="location", palette=CITY_COLORS, legend=False, ax=axes[0])
axes[0].axhline(0, color="grey", linestyle="--", linewidth=1)
axes[0].set_title("Feels-like minus actual temperature")
axes[0].set_xlabel("")
axes[0].set_ylabel("Gap (°C)")

sns.scatterplot(data=weather, x="humidity", y="feels_gap", hue="location", palette=CITY_COLORS, alpha=0.6, ax=axes[1])
axes[1].axhline(0, color="grey", linestyle="--", linewidth=1)
axes[1].set_title("Humidity vs feels-like gap")
axes[1].set_xlabel("Humidity (%)")
axes[1].set_ylabel("Gap (°C)")
plt.tight_layout()
plt.show()

weather.groupby("location")[["humidity", "feels_gap"]].mean().round(2)

## 4. Daily temperature cycle (local time)

The average temperature for each **local** hour of the day. We need at least about one full day of data for this to be meaningful.

In [ ]:
hourly = weather.groupby(["location", "local_hour"])["temp"].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=hourly, x="local_hour", y="temp", hue="location", palette=CITY_COLORS, marker="o", ax=ax)
ax.set_xticks(range(0, 24, 2))
ax.set_title("Average temperature by local hour of day")
ax.set_xlabel("Local hour")
ax.set_ylabel("Temperature (°C)")
plt.show()

swing = weather.groupby("location")["temp"].agg(lambda s: s.max() - s.min()).round(1)
print("Temperature range (max - min) over the collected period:")
print(swing.to_string())

## 5. Wind and pressure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.violinplot(data=weather, x="location", y="wind", hue="location", palette=CITY_COLORS, legend=False, ax=axes[0], cut=0)
axes[0].set_title("Wind speed distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Wind speed (m/s)")

for city, g in weather.groupby("location"):
    axes[1].plot(g["forecastdate"], g["pressure"], label=city, color=CITY_COLORS.get(city))
axes[1].set_title("Atmospheric pressure over time")
axes[1].set_xlabel("Report time (UTC)")
axes[1].set_ylabel("Pressure (hPa)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Day length

`sunrise` and `sunset` are Unix timestamps. Ho Chi Minh City is near the equator, so its day length stays
close to 12 hours all year. Tokyo is further north, so its days are clearly shorter in winter and longer in summer.

In [ ]:
latest = weather.sort_values("forecastdate").groupby("location").tail(1).copy()
latest["day_length_h"] = (latest["sunset"] - latest["sunrise"]) / 3600
latest["sunrise_local"] = pd.to_datetime(latest["sunrise"], unit="s") + pd.to_timedelta(latest["location"].map(UTC_OFFSET), unit="h")
latest["sunset_local"] = pd.to_datetime(latest["sunset"], unit="s") + pd.to_timedelta(latest["location"].map(UTC_OFFSET), unit="h")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(latest["location"], latest["day_length_h"], color=[CITY_COLORS.get(c) for c in latest["location"]])
ax.axvline(12, color="grey", linestyle="--", linewidth=1, label="12 h")
for y, v in enumerate(latest["day_length_h"]):
    ax.text(v + 0.1, y, "{:.1f} h".format(v), va="center")
ax.set_xlim(0, 15)
ax.set_xlabel("Hours of daylight")
ax.set_title("Day length (latest report)")
ax.legend()
plt.show()

latest["day_length_h"] = latest["day_length_h"].round(2)
latest[["location", "sunrise_local", "sunset_local", "day_length_h"]]

## 7. Weather conditions

How often each condition (as described by OpenWeatherMap) was reported in each city.

In [ ]:
conditions = (weather.groupby(["location", "description"]).size()
              .groupby(level=0).transform(lambda s: s / s.sum() * 100)
              .rename("share_pct").reset_index())

g = sns.catplot(data=conditions, kind="bar", y="description", x="share_pct", col="location",
                hue="location", palette=CITY_COLORS, legend=False, height=4.5, aspect=1.3, sharey=False)
g.set_axis_labels("Share of reports (%)", "")
g.set_titles("{col_name}")
g.fig.suptitle("Reported weather conditions", y=1.04)
plt.show()

## 8. Correlation between weather variables

The correlations are computed separately for each city, because combining two very different climates would hide the patterns within each one.

In [ ]:
cols = ["temp", "feels_like", "humidity", "pressure", "wind"]
cities = weather["location"].unique()
fig, axes = plt.subplots(1, len(cities), figsize=(7 * len(cities), 5.5))
axes = np.atleast_1d(axes)
for ax, city in zip(axes, cities):
    corr = weather.loc[weather["location"] == city, cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True, ax=ax, cbar=False)
    ax.set_title(city)
plt.tight_layout()
plt.show()